# 02 · Storm table (track + intensity models)
Joins the storm skeleton (history features + targets from IBTrACS) with the storm-centred ERA5 features from
notebook 01. One row = one storm every 3 h. All targets live in this one table:
`y_dlat_L`, `y_dlon_L` (track), `y_dwind_L` (intensity), `y_ri` (rapid intensification), `y_peak_wind`, `y_peak_grade`.

**Runtime:** any CPU. **Takes:** under a minute. **Output:** `outputs/tables/storm_fixes`

In [ ]:
# ▶ Setup: connect Google Drive, install missing packages, load the project code
import sys, os, subprocess
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "catboost", "lightgbm", "xgboost"], check=False)

PROJECT_DIR = "/content/drive/MyDrive/CycloneProject"      # <- change only if you put the project elsewhere
if PROJECT_DIR not in sys.path:
    sys.path.insert(0, PROJECT_DIR)

import numpy as np, pandas as pd
from src import config as C
pd.set_option("display.width", 160); pd.set_option("display.max_columns", 60)
print("project:", C.PROJECT, "| ERA5:", C.ERA5_ROOT)

In [ ]:
from src.common import load_table, save_table, list_tables, load_many
from src.models import storm_feature_columns
rows = load_table(C.out("tables") / "storm_rows")
env = load_many(list_tables(C.out("tables", "storm_env")))
df = rows.merge(env, on=["SID", "time"], how="inner")
missing = len(rows) - len(df)
print(f"{len(df)} storm rows with ERA5 features ({missing} rows dropped: their month has no ERA5 table yet)")
save_table(df, C.out("tables") / "storm_fixes")
feats = storm_feature_columns(df)
print(f"{len(feats)} feature columns")

In [ ]:
# ▶ Quick look
print(df.groupby("split").agg(rows=("SID", "size"), storms=("SID", "nunique")))
ts = df[df.split == "test"]
print("\ntest storms by group (main = TEST years, recent = RECENT_TEST_MONTHS):")
display(ts.groupby(["test_group", "SID"]).agg(name=("NAME", "first"), start=("time", "min"), rows=("time", "size"),
                                            peak_kt=("wind", "max")).sort_values("start"))
tcols = [c for c in df.columns if c.startswith("y_")]
display(df[tcols].notna().mean().rename("share with target").to_frame().T.round(2))
print("correlation wind vs pres_drop:", round(df[["wind", "pres_drop"]].corr().iloc[0, 1], 2), "(expect strongly positive)")
print("RI cases:", int(df.y_ri.sum()), "of", int(df.y_ri.notna().sum()))
display(df[feats].describe().T[["mean", "std", "min", "max"]].round(2).head(40))

**Next:** `04_train_track_intensity.ipynb` (can run while `03` trains).